# Evaluate Fine-tuned AlphaGenome Model (2026-02-23_08-36-57)

This notebook evaluates the fine-tuned AlphaGenome model trained on D1MSN H3K27ac CUT&Tag data.

**Training Details:**
- **Checkpoint**: `checkpoints/fine_tuned_2026-02-23_08-36-57/best_checkpoint`
- **Signal**: D1MSN H3K27ac CUT&Tag (averaged, normalized to 100M insertions)
- **Strategy**: head_replace (new ATAC output head)
- **Windows**: 20,809 training windows (131,072 bp each)
- **Organism**: Mouse (Mus musculus)

**Evaluation Goals:**
1. Load the fine-tuned model
2. Generate predictions on held-out test intervals
3. Compute correlation with ground truth 
4. Visualize predictions vs targets across multiple genomic regions

## 1. Setup and Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

# Add project root to path
sys.path.insert(0, '/home/ray/AG_Finetune')

# TensorFlow: CPU only (avoid GPU conflicts with JAX)
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
tf.config.set_visible_devices([], 'TPU')

import jax
import jax.numpy as jnp
import haiku as hk
from alphagenome_research.model import dna_model
from alphagenome_research.model.metadata import metadata as metadata_lib
from alphagenome.data import genome
import orbax.checkpoint as ocp

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"Using device: {jax.devices()[0]}")

2026-02-25 06:58:41.491496: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-25 06:58:41.548552: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/ray/miniforge3/envs/alphagenome/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.9.0
JAX devices: [CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3), CudaDevice(id=4), CudaDevice(id=5), CudaDevice(id=6), CudaDevice(id=7)]
Using device: cuda:0


## 2. Configuration

In [2]:
# Paths
CHECKPOINT_DIR = '/home/ray/AG_Finetune/checkpoints/fine_tuned_2026-02-23_08-36-57/best_checkpoint'
SIGNAL_PATH = '/home/ray/AG_Finetune/data/CUTTag/processed_bedgraph/D1MSN_H3K27ac_averaged_normalized.bedgraph'
FASTA_PATH = '/home/ray/AG_Finetune/data/resources/mm10.fa'

# Model config
ORGANISM = dna_model.Organism.MUS_MUSCULUS
SEQUENCE_LENGTH = 131072  # 2^17 bp
ORGANISM_ID = 1  # Mouse

# Test regions (held-out chromosomes or specific intervals)
# Using chr19 as a typical autosome for testing
TEST_INTERVALS = [
    genome.Interval(chromosome='chr19', start=5_000_000, end=5_131_072),
    genome.Interval(chromosome='chr19', start=10_000_000, end=10_131_072),
    genome.Interval(chromosome='chr19', start=20_000_000, end=20_131_072),
    genome.Interval(chromosome='chr19', start=30_000_000, end=30_131_072),
    genome.Interval(chromosome='chr19', start=40_000_000, end=40_131_072),
    # Add chr18 for diversity
    genome.Interval(chromosome='chr18', start=10_000_000, end=10_131_072),
    genome.Interval(chromosome='chr18', start=30_000_000, end=30_131_072),
    genome.Interval(chromosome='chr18', start=50_000_000, end=50_131_072),
]

print(f"Checkpoint: {CHECKPOINT_DIR}")
print(f"Signal file: {SIGNAL_PATH}")
print(f"Organism: {ORGANISM}")
print(f"Sequence length: {SEQUENCE_LENGTH:,} bp")
print(f"Number of test intervals: {len(TEST_INTERVALS)}")

Checkpoint: /home/ray/AG_Finetune/checkpoints/fine_tuned_2026-02-23_08-36-57/best_checkpoint
Signal file: /home/ray/AG_Finetune/data/CUTTag/processed_bedgraph/D1MSN_H3K27ac_averaged_normalized.bedgraph
Organism: Organism.MUS_MUSCULUS
Sequence length: 131,072 bp
Number of test intervals: 8


## 3. Load Ground Truth Signal

Load the CUT&Tag signal data for comparison with predictions.

In [3]:
def load_bedgraph_signal(bedgraph_path, chrom_sizes_path='/home/ray/AG_Finetune/data/resources/mm10.chrom.sizes'):
    """Load bedgraph signal into memory."""
    # Load chromosome sizes
    chrom_sizes = pd.read_csv(chrom_sizes_path, sep='\t', header=None, names=['chrom', 'size'])
    sizes = dict(zip(chrom_sizes['chrom'], chrom_sizes['size']))
    
    # Initialize signal arrays
    signal = {chrom: np.zeros(size, dtype=np.float32) for chrom, size in sizes.items()}
    
    # Load bedgraph
    print(f"Loading bedgraph from {bedgraph_path}...")
    with open(bedgraph_path) as f:
        for i, line in enumerate(f):
            if line.startswith('track') or line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            chrom, start, end, value = parts[0], int(parts[1]), int(parts[2]), float(parts[3])
            if chrom in signal:
                signal[chrom][start:end] = value
            if (i + 1) % 1_000_000 == 0:
                print(f"  Processed {i+1:,} lines...")
    
    print(f"✓ Loaded signal for {len(signal)} chromosomes")
    return signal

# Load ground truth signal
ground_truth_signal = load_bedgraph_signal(SIGNAL_PATH)

# Check signal statistics
all_values = np.concatenate([signal for signal in ground_truth_signal.values()])
nonzero_values = all_values[all_values > 0]
print(f"\nSignal statistics:")
print(f"  Total bases: {len(all_values):,}")
print(f"  Non-zero bases: {len(nonzero_values):,} ({100*len(nonzero_values)/len(all_values):.2f}%)")
print(f"  Mean (all): {all_values.mean():.6f}")
print(f"  Mean (non-zero): {nonzero_values.mean():.6f}")
print(f"  Max: {all_values.max():.6f}")
print(f"  Median (non-zero): {np.median(nonzero_values):.6f}")

Loading bedgraph from /home/ray/AG_Finetune/data/CUTTag/processed_bedgraph/D1MSN_H3K27ac_averaged_normalized.bedgraph...
  Processed 1,000,000 lines...
  Processed 2,000,000 lines...
  Processed 3,000,000 lines...
  Processed 4,000,000 lines...
  Processed 5,000,000 lines...
  Processed 6,000,000 lines...
  Processed 7,000,000 lines...
  Processed 8,000,000 lines...
  Processed 9,000,000 lines...
  Processed 10,000,000 lines...
  Processed 11,000,000 lines...
  Processed 12,000,000 lines...
  Processed 13,000,000 lines...
  Processed 14,000,000 lines...
  Processed 15,000,000 lines...
  Processed 16,000,000 lines...
  Processed 17,000,000 lines...
  Processed 18,000,000 lines...
  Processed 19,000,000 lines...
  Processed 20,000,000 lines...
  Processed 21,000,000 lines...
  Processed 22,000,000 lines...
  Processed 23,000,000 lines...
  Processed 24,000,000 lines...
  Processed 25,000,000 lines...
  Processed 26,000,000 lines...
  Processed 27,000,000 lines...
  Processed 28,000,000 l

## 4. Load Fine-tuned Model

Load the fine-tuned AlphaGenome model with the new ATAC output head.

In [8]:
import orbax.checkpoint as ocp

# Load checkpoint manually
checkpointer = ocp.StandardCheckpointer()
checkpoint_data = checkpointer.restore(CHECKPOINT_DIR)

# Extract only the model parameters (not optimizer state)
params = checkpoint_data['params']  # or checkpoint_data[0] if it's a tuple

# Now create the model with just the params
model = dna_model.create(
    checkpoint_path=None,  # Don't load from checkpoint path
    organism_settings={
        ORGANISM: dna_model.OrganismSettings(
            metadata=output_metadata,
        )
    },
    device=jax.devices()[0],
)

# Replace the model's params with your fine-tuned params
model._params = jax.device_put(params, jax.devices()[0])

print(f"✓ Model loaded successfully with fine-tuned parameters")
print(f"  Device: {model._device_context._device}")

TypeError: Invalid path type: None

In [7]:
# Define output metadata (matching training setup)
track_df = pd.DataFrame([{
    'output_type': 'ATAC',
    'name': 'D1MSN H3K27ac CUT&Tag',
    'strand': '.',
    'file_path': SIGNAL_PATH,
}])
output_metadata = metadata_lib.AlphaGenomeOutputMetadata(atac=track_df)

# Load model with fine-tuned checkpoint
print(f"Loading fine-tuned model from {CHECKPOINT_DIR}...")
model = dna_model.create(
    checkpoint_path=CHECKPOINT_DIR,
    organism_settings={
        ORGANISM: dna_model.OrganismSettings(
            metadata=output_metadata,
        )
    },
    device=jax.devices()[0],
)
print(f"✓ Model loaded successfully")
print(f"  Device: {model._device_context._device}")

Loading fine-tuned model from /home/ray/AG_Finetune/checkpoints/fine_tuned_2026-02-23_08-36-57/best_checkpoint...


TypeError: Argument 'opt_state' of type <class 'str'> is not a valid JAX type

## 5. Generate Predictions on Test Intervals

Run the fine-tuned model on held-out test intervals and collect predictions.

In [ ]:
def extract_ground_truth(interval, signal_dict):
    """Extract ground truth signal for a given interval."""
    chrom = interval.chromosome
    start = interval.start
    end = interval.end
    if chrom not in signal_dict:
        return np.zeros(end - start, dtype=np.float32)
    return signal_dict[chrom][start:end]

# Generate predictions for all test intervals
results = []

print("Generating predictions on test intervals...")
for i, interval in enumerate(TEST_INTERVALS):
    print(f"\n[{i+1}/{len(TEST_INTERVALS)}] Processing {interval.chromosome}:{interval.start:,}-{interval.end:,}")
    
    try:
        # Generate prediction
        preds = model.predict_interval(
            interval,
            requested_outputs=[dna_model.OutputType.ATAC],
            ontology_terms=None,
        )
        
        # Extract prediction values (1bp resolution)
        pred_values = preds.atac.scaled_predictions_1bp.values[0, :]  # Shape: [seq_len]
        
        # Extract ground truth
        true_values = extract_ground_truth(interval, ground_truth_signal)
        
        # Ensure same length
        min_len = min(len(pred_values), len(true_values))
        pred_values = pred_values[:min_len]
        true_values = true_values[:min_len]
        
        # Compute correlations
        pearson_r, pearson_p = pearsonr(pred_values, true_values)
        spearman_r, spearman_p = spearmanr(pred_values, true_values)
        
        # Compute MSE
        mse = np.mean((pred_values - true_values) ** 2)
        
        results.append({
            'interval': f"{interval.chromosome}:{interval.start:,}-{interval.end:,}",
            'chromosome': interval.chromosome,
            'start': interval.start,
            'end': interval.end,
            'predictions': pred_values,
            'targets': true_values,
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'spearman_r': spearman_r,
            'spearman_p': spearman_p,
            'mse': mse,
        })
        
        print(f"  Pearson r: {pearson_r:.4f} (p={pearson_p:.2e})")
        print(f"  Spearman r: {spearman_r:.4f} (p={spearman_p:.2e})")
        print(f"  MSE: {mse:.6f}")
        
    except Exception as e:
        print(f"  ⚠️  Error: {e}")
        continue

print(f"\n✓ Completed predictions for {len(results)} intervals")

## 6. Correlation Analysis Summary

Aggregate correlation statistics across all test intervals.

In [ ]:
# Create summary DataFrame
summary_df = pd.DataFrame([{
    'interval': r['interval'],
    'chromosome': r['chromosome'],
    'pearson_r': r['pearson_r'],
    'spearman_r': r['spearman_r'],
    'mse': r['mse'],
} for r in results])

print("\n" + "="*80)
print("CORRELATION SUMMARY")
print("="*80)
print(f"\nOverall Statistics (N={len(summary_df)} intervals):")
print(f"  Pearson correlation:  {summary_df['pearson_r'].mean():.4f} ± {summary_df['pearson_r'].std():.4f}")
print(f"    Range: [{summary_df['pearson_r'].min():.4f}, {summary_df['pearson_r'].max():.4f}]")
print(f"  Spearman correlation: {summary_df['spearman_r'].mean():.4f} ± {summary_df['spearman_r'].std():.4f}")
print(f"    Range: [{summary_df['spearman_r'].min():.4f}, {summary_df['spearman_r'].max():.4f}]")
print(f"  MSE: {summary_df['mse'].mean():.6f} ± {summary_df['mse'].std():.6f}")

print("\nPer-interval breakdown:")
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_path = CHECKPOINT_DIR.replace('/best_checkpoint', '/evaluation_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"\n✓ Saved summary to {summary_path}")

## 7. Visualization: Correlation Distributions

Plot distribution of correlation coefficients across test intervals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pearson correlation
axes[0].hist(summary_df['pearson_r'], bins=20, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(summary_df['pearson_r'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f"Mean: {summary_df['pearson_r'].mean():.4f}")
axes[0].set_xlabel('Pearson Correlation (r)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Pearson Correlations', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Spearman correlation
axes[1].hist(summary_df['spearman_r'], bins=20, alpha=0.7, color='darkorange', edgecolor='black')
axes[1].axvline(summary_df['spearman_r'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean: {summary_df['spearman_r'].mean():.4f}")
axes[1].set_xlabel('Spearman Correlation (ρ)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Spearman Correlations', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR.replace('/best_checkpoint', '/correlation_distributions.png'), dpi=300, bbox_inches='tight')
plt.show()

## 8. Visualization: Predictions vs Targets

Visualize predictions against ground truth for selected intervals.

In [ ]:
# Select top 3 intervals by Pearson correlation
top_intervals_idx = summary_df.nlargest(3, 'pearson_r').index.tolist()

fig, axes = plt.subplots(3, 1, figsize=(16, 10))

for plot_idx, result_idx in enumerate(top_intervals_idx):
    result = results[result_idx]
    ax = axes[plot_idx]
    
    # Subsample for visualization (every 100bp to avoid overcrowding)
    subsample = 100
    x = np.arange(0, len(result['predictions']), subsample) / 1000  # Convert to kb
    pred_sub = result['predictions'][::subsample]
    true_sub = result['targets'][::subsample]
    
    ax.plot(x, true_sub, label='Ground Truth', color='red', alpha=0.7, linewidth=1.5)
    ax.plot(x, pred_sub, label='Prediction', color='blue', alpha=0.7, linewidth=1.5)
    
    ax.set_xlabel('Position (kb)', fontsize=11)
    ax.set_ylabel('Signal', fontsize=11)
    ax.set_title(
        f"{result['interval']} | Pearson r={result['pearson_r']:.4f}, Spearman ρ={result['spearman_r']:.4f}",
        fontsize=12, fontweight='bold'
    )
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR.replace('/best_checkpoint', '/predictions_vs_targets_top3.png'), dpi=300, bbox_inches='tight')
plt.show()

## 9. Scatter Plot: All Predictions vs Targets

Create a scatter plot of all predicted vs true values across all intervals.

In [ ]:
# Concatenate all predictions and targets
all_predictions = np.concatenate([r['predictions'] for r in results])
all_targets = np.concatenate([r['targets'] for r in results])

# Subsample for visualization if too many points
max_points = 50000
if len(all_predictions) > max_points:
    indices = np.random.choice(len(all_predictions), max_points, replace=False)
    plot_predictions = all_predictions[indices]
    plot_targets = all_targets[indices]
else:
    plot_predictions = all_predictions
    plot_targets = all_targets

# Compute overall correlation
overall_pearson, _ = pearsonr(all_predictions, all_targets)
overall_spearman, _ = spearmanr(all_predictions, all_targets)

# Create scatter plot
fig, ax = plt.subplots(figsize=(10, 10))

# Hexbin plot for density
hexbin = ax.hexbin(plot_targets, plot_predictions, gridsize=50, cmap='Blues', mincnt=1, alpha=0.8)
cb = plt.colorbar(hexbin, ax=ax, label='Density')

# Add diagonal line (perfect prediction)
max_val = max(plot_targets.max(), plot_predictions.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')

ax.set_xlabel('Ground Truth Signal', fontsize=13)
ax.set_ylabel('Predicted Signal', fontsize=13)
ax.set_title(
    f'Predictions vs Ground Truth\nPearson r={overall_pearson:.4f}, Spearman ρ={overall_spearman:.4f}\n(N={len(all_predictions):,} positions, {len(results)} intervals)',
    fontsize=14, fontweight='bold'
)
ax.legend(loc='upper left', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR.replace('/best_checkpoint', '/scatter_predictions_vs_targets.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Overall Pearson correlation across all positions: {overall_pearson:.4f}")
print(f"✓ Overall Spearman correlation across all positions: {overall_spearman:.4f}")

## 10. Per-Chromosome Correlation Analysis

Compare model performance across different chromosomes.

In [ ]:
# Group by chromosome
chrom_summary = summary_df.groupby('chromosome').agg({
    'pearson_r': ['mean', 'std', 'count'],
    'spearman_r': ['mean', 'std'],
    'mse': ['mean', 'std']
}).round(4)

print("\n" + "="*80)
print("PER-CHROMOSOME PERFORMANCE")
print("="*80)
print(chrom_summary)

# Bar plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

chrom_means = summary_df.groupby('chromosome').agg({'pearson_r': 'mean', 'spearman_r': 'mean'})

axes[0].bar(range(len(chrom_means)), chrom_means['pearson_r'], alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_xticks(range(len(chrom_means)))
axes[0].set_xticklabels(chrom_means.index, rotation=0)
axes[0].set_ylabel('Mean Pearson r', fontsize=12)
axes[0].set_xlabel('Chromosome', fontsize=12)
axes[0].set_title('Mean Pearson Correlation by Chromosome', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(range(len(chrom_means)), chrom_means['spearman_r'], alpha=0.7, color='darkorange', edgecolor='black')
axes[1].set_xticks(range(len(chrom_means)))
axes[1].set_xticklabels(chrom_means.index, rotation=0)
axes[1].set_ylabel('Mean Spearman ρ', fontsize=12)
axes[1].set_xlabel('Chromosome', fontsize=12)
axes[1].set_title('Mean Spearman Correlation by Chromosome', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR.replace('/best_checkpoint', '/performance_by_chromosome.png'), dpi=300, bbox_inches='tight')
plt.show()

## 11. Summary

**Model Evaluation Complete!**

Key findings:
- Evaluated fine-tuned AlphaGenome model on held-out test intervals
- Computed correlation metrics (Pearson and Spearman)
- Generated visualizations comparing predictions to ground truth

All results and figures have been saved to the checkpoint directory.